# Hidden Markov model and Q-learning experiments

One ticker is used per experiment. Market regimes are fitted on training dates only and assigned by forward filtering, so a regime at time *t* does not use subsequent observations. The trading policy is trained on historical data and evaluated separately without learning.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd().resolve()
if not (root / 'scripts').exists():
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))

from data_processing import load_data, prepare_features
from model_training import chronological_split, QuantileBinner, BASELINE_FEATURES
from hmm_integration import fit_hmm_on_indicators, infer_filtered_regimes
from rl_agent import QLearningAgent, SimpleTradingEnv


In [ ]:
raw = load_data(root / 'data/raw/all_stocks_2006-01-01_to_2018-01-01.csv')
stock = prepare_features(raw.loc[raw['Name'] == 'AAPL'])
split = chronological_split(stock)
print('Train/validation/test observations:', len(split.train), len(split.validation), len(split.test))


In [ ]:
hmm_model, train_regimes = fit_hmm_on_indicators(split.train, n_components=3)
test_regimes = infer_filtered_regimes(hmm_model, split.test)
print('Train-regime counts:', train_regimes['Market_Regime'].value_counts().to_dict())
print('Test-regime counts:', test_regimes['Market_Regime'].value_counts().to_dict())

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(test_regimes['Date'], test_regimes['Market_Regime'].astype(int), linewidth=0.8)
ax.set(title='Forward-filtered HMM regimes on later observations', xlabel='Date', ylabel='Regime')
plt.tight_layout()
plt.show()


In [ ]:
binner = QuantileBinner(4).fit(train_regimes, BASELINE_FEATURES)

def rl_frame(data):
    frame = data.copy()
    bins = binner.transform(frame, BASELINE_FEATURES)
    for column in BASELINE_FEATURES:
        frame[f'{column}_binned'] = bins[column].to_numpy()
    return frame

training = rl_frame(train_regimes.tail(250))
testing = rl_frame(test_regimes)
agent = QLearningAgent(alpha=0.1, gamma=0.95, epsilon=0.1, seed=42)
train_env = SimpleTradingEnv(training, transaction_cost=0.001)
rewards = [agent.simulate_episode(train_env) for _ in range(50)]

test_env = SimpleTradingEnv(testing, transaction_cost=0.001)
held_out_reward = agent.simulate_episode(test_env, train=False)
buy_and_hold = float(testing['Close'].iloc[-1] - testing['Close'].iloc[0] - testing['Close'].iloc[0] * 0.001)
print(f'Frozen policy holdout P/L, one-share dollar units: {held_out_reward:.2f}')
print(f'Buy-and-hold comparison, same units: {buy_and_hold:.2f}')
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(rewards)
ax.set(title='Q-learning training episodes (in-sample)', xlabel='Episode', ylabel='Profit/loss ($, one share)')
plt.tight_layout()
plt.show()


The HMM states are unlabeled statistical regimes; they do not inherently mean bull or bear markets. This is a simplified historical simulator without slippage, liquidity constraints, or position sizing beyond one share. Reported returns are experimental, not a live-performance claim.